# 📚 Topic Modeling: Descubrimiento de Tópicos Latentes con LDA

### Autor: Néstor León | Business Intelligence & Analytics
---

Cuando una compañía recibe miles de reseñas, reclamaciones o tickets de soporte al día, es imposible clasificarlos a mano uno a uno. El **modelado de tópicos probabilístico**, en particular **Latent Dirichlet Allocation (LDA)**, nos permite descubrir de forma automática qué temáticas subyacen en la conversación.

En este notebook:
1. Configuramos el algoritmo LDA en `scikit-learn`.
2. Modelamos la distribución de palabras por cada tópico latente ($eta$).
3. Asignamos a cada documento su vector probabilístico de pertenencia ($	heta$) para clasificarlo en un departamento de resolución (Facturación, Soporte Técnico, Calidad de Red, etc.).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

print("Librerías cargadas con éxito.")


## 1. Preparación del Corpus de Tickets de Soporte

Construimos un corpus representativo de incidencias operativas de clientes para segmentarlas por temática subyacente.


In [ ]:
tickets_soporte = [
    "No tengo conexion a internet desde ayer por la tarde, la luz del router parpadea en rojo.",
    "El importe de la factura de este mes es 30 euros superior a lo contratado, exijo explicaciones.",
    "La velocidad de descarga de la fibra optica es muy baja comparada con los 600 megas prometidos.",
    "Cobro indebido de un servicio premium que jamas active en mi linea movil.",
    "Quiero solicitar la baja de mi contrato de telefonia por cambio de domicilio.",
    "El tecnico vino a casa pero no soluciono el corte de la linea fija ni la television.",
    "Me llego una factura duplicada correspondiente al periodo de facturacion anterior.",
    "Perdida de senal continua en el telefono movil cada vez que entro en mi oficina.",
    "Excelente atencion comercial en tienda, me resolvieron todas las dudas sobre el cambio de tarifa.",
    "Problemas continuos con el wifi del router, se desconecta cada pocos minutos mientras teletrabajo.",
    "Cargo sorpresa por cargos de roaming que no corresponden a mi estancia fuera del pais.",
    "Caida general del servicio de fibra en toda la manzana de mi barrio."
]

df_tickets = pd.DataFrame({'ticket': tickets_soporte})
df_tickets.head(4)


## 2. Extracción de Características mediante Frecuencias

Para LDA necesitamos conteos de términos (no TF-IDF, ya que LDA es un modelo generativo basado en distribuciones multinomiales de palabras).


In [ ]:
stop_es = [
    'el', 'la', 'los', 'las', 'un', 'una', 'de', 'en', 'a', 'con', 'por', 'para', 'que', 'no', 'si',
    'mi', 'me', 'es', 'del', 'al', 'se', 'lo', 'este', 'esta', 'este', 'mes', 'dia', 'ayer', 'casa',
    'pero', 'mas', 'cada', 'toda', 'todo', 'tengo', 'vino', 'llego'
]

vectorizer = CountVectorizer(stop_words=stop_es, min_df=1, max_df=0.9, token_pattern=r'(?u)\b[a-zA-ZáéíóúÁÉÍÓÚñÑ]{3,}\b')
dtm = vectorizer.fit_transform(df_tickets['ticket'])
feature_names = vectorizer.get_feature_names_out()

print(f"Dimensiones de la matriz término-documento: {dtm.shape}")


## 3. Ajuste del Modelo Latent Dirichlet Allocation (LDA)

Fijamos $K = 3$ tópicos hipotéticos de negocio:
- **Tópico 0**: Facturación y cobros.
- **Tópico 1**: Conectividad, fibra y router (Soporte Técnico).
- **Tópico 2**: Cobertura móvil y gestión de contratos/bajas.


In [ ]:
lda_model = LatentDirichletAllocation(
    n_components=3,
    random_state=42,
    learning_method='online',
    max_iter=30
)

lda_model.fit(dtm)
print("Modelo LDA entrenado.")


### Inspección de las Palabras Más Probables por Tópico


In [ ]:
def mostrar_top_palabras_topico(model, feature_names, n_top_words=6):
    topic_data = {}
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        topic_data[f"Tópico {topic_idx + 1}"] = top_features
    return pd.DataFrame(topic_data)

df_top_words = mostrar_top_palabras_topico(lda_model, feature_names)
df_top_words


## 4. Clasificación y Asignación de Documentos a Tópicos

Calculamos la distribución de tópicos de cada ticket de soporte para enrutarlos automáticamente al equipo responsable.


In [ ]:
topic_distributions = lda_model.transform(dtm)
topicos_predichos = np.argmax(topic_distributions, axis=1)

df_tickets['topico_asignado'] = [f"Tópico {t + 1}" for t in topicos_predichos]
df_tickets['probabilidad_topico'] = [round(np.max(p), 3) for p in topic_distributions]

df_tickets[['ticket', 'topico_asignado', 'probabilidad_topico']].head(6)


## 5. Conclusiones de Negocio

1. **Automatización del Triaje**: Sin necesidad de reglas *hardcoded*, LDA separa con precisión los incidentes de red (fibra, router, wifi) de las disputas económicas (facturas, cobros indebidos).
2. **Escalabilidad**: Este pipeline puede procesar decenas de miles de tickets en cuestión de segundos, reduciendo drásticamente el tiempo medio de respuesta (*Time-to-Resolution*) al derivar el ticket directamente al departamento especialista.
